In [0]:
from pyspark.sql import functions as F

CATALOG = dbutils.widgets.get("catalog")
RAW_SCHEMA = dbutils.widgets.get("stream_schema")
TITLE = dbutils.widgets.get("title")
BRONZE_SCHEMA = f"{RAW_SCHEMA}_bronze"

BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.{TITLE}"

BASE_PATH = f"/Volumes/{CATALOG}/{RAW_SCHEMA}/raw/{RAW_SCHEMA}"

LANDING_PATH = f"{BASE_PATH}/landing"
SCHEMA_PATH = f"{BASE_PATH}/schemas/bronze"
BRONZE_CHECKPOINT = f"{BASE_PATH}/checkpoints/bronze"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")

In [0]:
stream_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", SCHEMA_PATH)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("rescuedDataColumn", "_rescued_data")
        .option("header", "true")
        .option("inferSchema", "false")
        .option("quote", '"')
        .option("escape", '"')
        .option("multiLine", "true")
        .option("mode", "PERMISSIVE")
        .load(LANDING_PATH)
        .withColumn("ingestion_time", F.current_timestamp())
        .withColumn("source_file", F.col("_metadata.file_path"))
)

In [0]:
query = (
    stream_df.writeStream
        .format("delta")
        .option("checkpointLocation", BRONZE_CHECKPOINT)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(BRONZE_TABLE)
)

query.awaitTermination()